In [1]:
%run start
%load_ext autoreload
%autoreload 2

Root set to: /home/bdudas/obesity_challange


In [2]:
from src.data.perturbation_data import get_loaders
from src.models.CycleTransformerv2 import CycleTransformer
from omegaconf import OmegaConf
import torch
import numpy as np
import pandas as pd
import anndata as ad
import tqdm as tqdm

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
modelwkgs = OmegaConf.load("configs/cycle_transformer.yaml")
cpkt_name = "corrected_cycle"
pretrained_path = f"misc/best_runs/{cpkt_name}/checkpoints/best-checkpoint.ckpt"
model = CycleTransformer(**modelwkgs.model_kwargs)
model.configure_cycle()
model.load_state_dict(torch.load(pretrained_path)["state_dict"])

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


<All keys matched successfully>

In [4]:
pertList = np.loadtxt("data/predict_perturbations.txt", dtype=str)

In [17]:
train_loader, val_loader,gene_to_idx, idx_to_gene = get_loaders("",batch_size=100, num_workers=0)

In [18]:
from itertools import cycle
biter = iter(cycle(train_loader))

In [7]:
def generate_predictions(model,x_ctrl,pert_indicies):
    z_ctrl = model.encoder(x_ctrl)
    pert_indices = pert_indicies.long()
    z_prompt = model.get_perturbation_prompt(x_ctrl,pert_indices)
    delta_fwd = model.transition_fwd(z_ctrl, z_prompt)
    z_fake_pert = z_ctrl + delta_fwd
    x_fake_pert = model.decoder(z_fake_pert)
    x_fake_pert = x_fake_pert.view(100,-1).detach().cpu()
    x_fake_pert = x_fake_pert[:,:-8]  # drop last 8
    
    logits_fake_pert = model.latentClassifier(z_fake_pert)
    logits_fake_pert = logits_fake_pert.detach().cpu()
    logits_argmaxes = torch.argmax(logits_fake_pert,dim=1)
    return x_fake_pert.numpy(), logits_argmaxes.numpy()

In [8]:
X_data = []
pred_prog_port = pd.DataFrame(columns=["gene","pre_adipo","adipo","lipo","other","lipo_adipo"])

In [9]:
class_names = ['pre_adipo', 'adipo', 'lipo', 'other']

In [48]:
X_data = []
pred_prog_port =[] #pd.DataFrame(columns=["gene","pre_adipo","adipo","lipo","other","lipo_adipo"])
obsData = np.array([]) #pd.DataFrame(columns=["gene","pre_adipo","adipo","lipo","other"])

for i, gene in tqdm.tqdm(enumerate(pertList), total=len(pertList)):
    _, x_ctrl, _, _, ctrl_state = next(biter)
    if x_ctrl.size(0) != 100:
        # Restart iterator or just break
        biter = iter(train_loader)
        _, x_ctrl, _, _, ctrl_state = next(biter)
    
    pert_idx = gene_to_idx[gene]
    pert_indicies = torch.tensor([pert_idx]*100)
    x_fake_pert, logits_fake_pert = generate_predictions(model,x_ctrl,pert_indicies)
    
    X_data.append(x_fake_pert)
    
    logits_onehot = np.zeros((logits_fake_pert.size,4))
    logits_onehot[np.arange(logits_fake_pert.size),logits_fake_pert] = 1
    genes = [gene]*x_ctrl.size(0)
    genes = np.array(genes).reshape(-1, 1)

    combined = np.hstack((genes, logits_onehot))
    obsData = np.vstack((obsData, combined)) if obsData.size else combined
    
    

100%|██████████| 2863/2863 [13:31<00:00,  3.53it/s]


In [53]:
obsData = obsData.reshape(-1,100,5)

In [55]:
class_names 

['pre_adipo', 'adipo', 'lipo', 'other']

In [56]:
obsDF = pd.DataFrame(obsData.reshape(-1,5), columns=["gene","pre_adipo","adipo","lipo","other"])

In [60]:
X_data = np.array(X_data)

In [66]:
X_Data = X_data.reshape(-1, X_data.shape[-1])

In [69]:
X_Data.shape, X_data.shape

((286300, 21592), (2863, 100, 21592))

In [70]:
submission_adata = ad.AnnData(X = X_Data, obs = obsDF)

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [75]:
submission_adata.write_h5ad("prediction.h5ad")